# PyTreeArrayDistribution Tutorial

`PyTreeArrayDistribution[T]` is an intermediate layer in ProbPipe's distribution hierarchy that sits between the fully generic `Distribution[T]` base class and the concrete `ArrayDistribution` specialization. It adds:

- **Pytree structure** (`treedef`) — describes the tree layout of a single sample.
- **Shape semantics** — `batch_shape`, per-leaf `event_shapes`, `flat_event_shapes`, and `flat_dim`.
- **Flatten/unflatten** — convert between structured pytree values and flat vectors.
- **Flat-view interop** — `as_flat_distribution()` returns a `FlattenedView` that wraps any `PyTreeArrayDistribution` as a standard `ArrayDistribution`, enabling seamless use with algorithms that expect flat vectors (MCMC, optimizers, VI, etc.).

All existing `ArrayDistribution` subclasses (Normal, MultivariateNormal, Gamma, etc.) automatically inherit the pytree interface because `ArrayDistribution` extends `PyTreeArrayDistribution[Array]`.

## The Distribution Hierarchy

ProbPipe's distribution classes form a three-level hierarchy:

```
Distribution[T]                  — generic base (sampling + optional log_prob)
  └── PyTreeArrayDistribution[T] — pytree structure, shapes, flatten/unflatten
        └── ArrayDistribution     — single-array specialization (T = Array)
              └── TFPDistribution  — delegates to tfd.* (Normal, Gamma, MVN, ...)
```

Every concrete distribution is an instance of all three base classes.

In [ ]:
import jax
import jax.numpy as jnp
from probpipe import (
    Distribution, PyTreeArrayDistribution, ArrayDistribution,
    FlattenedView, Normal, MultivariateNormal, Gamma,
    EmpiricalDistribution, Provenance, real, positive,
    monte_carlo, BootstrapDistribution,
)

key = jax.random.PRNGKey(0)

n = Normal(loc=0.0, scale=1.0)
print(f"isinstance Normal of Distribution:              {isinstance(n, Distribution)}")
print(f"isinstance Normal of PyTreeArrayDistribution:   {isinstance(n, PyTreeArrayDistribution)}")
print(f"isinstance Normal of ArrayDistribution:         {isinstance(n, ArrayDistribution)}")

## Shape Semantics

`PyTreeArrayDistribution` exposes a rich set of shape descriptors:

| Property | Description |
|---|---|
| `treedef` | Pytree structure of a single sample |
| `batch_shape` | Shape of independent parameter batches (shared across leaves) |
| `event_shape` | Shape of a single draw (for `ArrayDistribution`) |
| `event_shapes` | Per-leaf event shapes as a pytree |
| `flat_event_shapes` | Event shapes as a flat list in canonical leaf order |
| `flat_dim` | Total dimensionality of the flattened representation |

Let's see how these differ for scalar, multivariate, and batched distributions.

In [ ]:
scalar = Normal(loc=0.0, scale=1.0)
mvn = MultivariateNormal(loc=jnp.zeros(3), cov=jnp.eye(3))
batched = Normal(loc=jnp.array([0.0, 1.0, 2.0]), scale=1.0)

for name, d in [("Scalar Normal", scalar), ("MVN(3)", mvn), ("Batched Normal", batched)]:
    print(f"{name}:")
    print(f"  treedef           = {d.treedef}")
    print(f"  batch_shape       = {d.batch_shape}")
    print(f"  event_shape       = {d.event_shape}")
    print(f"  event_shapes      = {d.event_shapes}")
    print(f"  flat_event_shapes = {d.flat_event_shapes}")
    print(f"  flat_dim          = {d.flat_dim}")
    print()

## Flatten and Unflatten Values

`flatten_value()` converts a pytree sample into a flat 1D vector by raveling each leaf and concatenating in canonical leaf order. `unflatten_value()` reverses the process. Together they provide a lossless roundtrip.

Both methods handle batched values automatically — extra leading dimensions are preserved.

In [ ]:
# Single sample roundtrip
mvn = MultivariateNormal(loc=jnp.zeros(3), cov=jnp.eye(3))
sample = mvn.sample(jax.random.PRNGKey(1))
print(f"Original sample shape: {sample.shape}")

flat = mvn.flatten_value(sample)
print(f"Flattened shape:       {flat.shape}")

restored = mvn.unflatten_value(flat)
print(f"Restored shape:        {restored.shape}")
print(f"Roundtrip match:       {jnp.allclose(sample, restored)}")

In [ ]:
# Batched roundtrip
samples = mvn.sample(jax.random.PRNGKey(2), sample_shape=(100,))
print(f"Batched samples shape:  {samples.shape}")

flat_batch = mvn.flatten_value(samples)
print(f"Flattened batch shape:  {flat_batch.shape}")

restored_batch = mvn.unflatten_value(flat_batch)
print(f"Restored batch shape:   {restored_batch.shape}")
print(f"Batched roundtrip match: {jnp.allclose(samples, restored_batch)}")

## FlattenedView: Converting to Flat Distributions

`FlattenedView` wraps any `PyTreeArrayDistribution` as a flat `ArrayDistribution` with `event_shape=(flat_dim,)`. You obtain one via `dist.as_flat_distribution()`.

This is the primary interoperability mechanism: any algorithm written for `ArrayDistribution` (MCMC samplers, variational inference, optimizers) works with `PyTreeArrayDistribution` distributions through this view.

In [ ]:
mvn = MultivariateNormal(loc=jnp.array([1.0, -1.0, 0.5]), cov=jnp.eye(3))
flat_dist = mvn.as_flat_distribution()

print(f"Type:                      {type(flat_dist).__name__}")
print(f"isinstance ArrayDistribution: {isinstance(flat_dist, ArrayDistribution)}")
print(f"event_shape:               {flat_dist.event_shape}")
print(f"batch_shape:               {flat_dist.batch_shape}")
print(f"base_distribution type:    {type(flat_dist.base_distribution).__name__}")
print(f"repr:                      {flat_dist}")

In [ ]:
# Sampling and log_prob consistency between MVN and its FlattenedView
k1, k2 = jax.random.split(jax.random.PRNGKey(3))

# Sample from the flat view
flat_samples = flat_dist.sample(k1, sample_shape=(5,))
print(f"Flat samples shape: {flat_samples.shape}")

# Log-prob consistency
original_sample = mvn.sample(k2)
flat_sample = mvn.flatten_value(original_sample)

lp_original = mvn.log_prob(original_sample)
lp_flat = flat_dist.log_prob(flat_sample)
print(f"\nlog_prob from MVN:           {float(lp_original):.6f}")
print(f"log_prob from FlattenedView: {float(lp_flat):.6f}")
print(f"Match: {jnp.allclose(lp_original, lp_flat)}")

In [ ]:
# unflatten_sample convenience method
flat_s = flat_dist.sample(jax.random.PRNGKey(4))
restored_s = flat_dist.unflatten_sample(flat_s)
print(f"Flat sample:     {flat_s}")
print(f"Unflattened:     {restored_s}")
print(f"Shape restored:  {restored_s.shape}")

## Provenance Tracking

Provenance works seamlessly through the hierarchy. When a distribution is created via `from_distribution()`, a `Provenance` object is attached as `.source`, recording the operation and parent distributions.

In [ ]:
g = Gamma(concentration=4.0, rate=2.0, name="prior_gamma")
emp = EmpiricalDistribution.from_distribution(
    g, key=jax.random.PRNGKey(5), num_samples=500
)
print(f"Source:  {emp.source}")
print(f"Parent:  {[type(p).__name__ for p in emp.source.parents]}")
print(f"isinstance PyTreeArrayDistribution: {isinstance(emp, PyTreeArrayDistribution)}")

## Expectations and the @monte_carlo Decorator

The `expectation()` method and `@monte_carlo`-decorated methods (like `mean()`) work through the full hierarchy. They operate on `Distribution[T]` and are inherited by `PyTreeArrayDistribution` and `ArrayDistribution` alike.

Expectations also work on `FlattenedView` instances, since they are `ArrayDistribution` subclasses.

In [ ]:
mvn = MultivariateNormal(loc=jnp.array([2.0, -1.0]), cov=jnp.eye(2))

# E[X] via the @monte_carlo-decorated mean()
mean_est = mvn.mean()
print(f"mean() = {mean_est}")

# E[||X||^2] via expectation()
norm_sq = mvn.expectation(
    lambda x: jnp.sum(x**2),
    key=jax.random.PRNGKey(6),
    num_evaluations=5000,
)
print(f"\nE[||X||^2] estimate: {float(norm_sq.mean()):.4f}")
print(f"  Exact value:       {float(jnp.sum(mvn.mean()**2) + jnp.trace(jnp.eye(2))):.4f}")
print(f"  Returns:           {type(norm_sq).__name__}")

# Works on FlattenedView too
flat_dist = mvn.as_flat_distribution()
flat_mean = flat_dist.expectation(
    lambda x: x,
    key=jax.random.PRNGKey(7),
    num_evaluations=5000,
    return_dist=False,
)
print(f"\nFlattenedView E[X]: {flat_mean}")

## Support Constraints

Each `ArrayDistribution` declares a `.support` property describing the set of values where the density is non-zero (e.g., `real`, `positive`, `unit_interval`). The `.supports` property provides per-leaf constraints for the pytree interface (for single-array distributions it equals `.support`).

`FlattenedView` always reports `real` support, since flattening maps any structured constraint to the real line.

In [ ]:
n = Normal(loc=0.0, scale=1.0)
g = Gamma(concentration=2.0, rate=1.0)

print(f"Normal support:   {n.support}")
print(f"Normal supports:  {n.supports}  (same as support for ArrayDistribution)")
print(f"Gamma support:    {g.support}")

# FlattenedView always has 'real' support
flat_n = n.as_flat_distribution()
flat_g = g.as_flat_distribution()
print(f"\nFlattenedView(Normal) support: {flat_n.support}")
print(f"FlattenedView(Gamma) support:  {flat_g.support}")

# Constraint checking
sample = g.sample(jax.random.PRNGKey(8))
print(f"\nGamma sample: {float(sample):.4f}")
print(f"Satisfies positive: {bool(positive.check(sample))}")
print(f"Satisfies real:     {bool(real.check(sample))}")

## Sampling and log_prob

Let's visualize how `MultivariateNormal` and its `FlattenedView` produce identical distributions. We draw samples from both views and compare their scatter plots and density contours.

In [ ]:
import matplotlib.pyplot as plt

mvn = MultivariateNormal(
    loc=jnp.array([1.0, -0.5]),
    cov=jnp.array([[1.0, 0.6], [0.6, 1.0]]),
)
flat_dist = mvn.as_flat_distribution()

# Draw samples from both views
k1, k2 = jax.random.split(jax.random.PRNGKey(9))
orig_samples = mvn.sample(k1, (500,))
flat_samples = flat_dist.sample(k2, (500,))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.scatter(orig_samples[:, 0], orig_samples[:, 1], alpha=0.3, s=5)
ax1.set_title("MultivariateNormal.sample()")
ax1.set_xlabel("x0")
ax1.set_ylabel("x1")
ax1.set_aspect("equal")

ax2.scatter(flat_samples[:, 0], flat_samples[:, 1], alpha=0.3, s=5)
ax2.set_title("FlattenedView.sample()")
ax2.set_xlabel("flat[0]")
ax2.set_ylabel("flat[1]")
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate log_prob on a grid and compare densities
x = jnp.linspace(-2, 4, 80)
y = jnp.linspace(-3, 2, 80)
X, Y = jnp.meshgrid(x, y)
points = jnp.stack([X.ravel(), Y.ravel()], axis=-1)

lp = jax.vmap(mvn.log_prob)(points).reshape(X.shape)
lp_flat = jax.vmap(flat_dist.log_prob)(points).reshape(X.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.contourf(X, Y, jnp.exp(lp), levels=20, cmap="viridis")
ax1.set_title("MVN density")
ax1.set_xlabel("x0")
ax1.set_ylabel("x1")

ax2.contourf(X, Y, jnp.exp(lp_flat), levels=20, cmap="viridis")
ax2.set_title("FlattenedView density (identical)")
ax2.set_xlabel("flat[0]")
ax2.set_ylabel("flat[1]")

plt.tight_layout()
plt.show()

print(f"Max |log_prob difference|: {float(jnp.max(jnp.abs(lp - lp_flat))):.2e}")

## Summary

Key takeaways from this notebook:

1. **`PyTreeArrayDistribution`** adds pytree structure and shape semantics between `Distribution[T]` and `ArrayDistribution`.
2. **All existing `ArrayDistribution` subclasses** (Normal, MVN, Gamma, etc.) automatically inherit the pytree interface.
3. **`flatten_value` / `unflatten_value`** convert between pytree and flat representations with a lossless roundtrip.
4. **`FlattenedView`** via `as_flat_distribution()` enables algorithm interoperability — any code expecting `ArrayDistribution` works seamlessly.
5. **Provenance, expectations, `@monte_carlo`, constraints** all work through the hierarchy without modification.
6. This is the **foundation for Phase 3+** — joint/product distributions over pytrees, `RandomMeasure`, and `RandomFunction`.